In [1]:
import pandas as pd
import numpy as np
import xarray as xr
import glob
import yaml
import os
import matplotlib.pyplot as plt
import pypsa

path = '/Users/mick/Documents/GitHub/masterthesis-mick/Wetterdaten'

In [18]:
## Turbine in use: NREL Reference Turbine (Offshore)
name = 'NREL_ReferenceTurbine_2020ATB_5.5MW'

# Load the turbine configuration (power curve, wind speeds, rated power, hub height) from YAML
with open(str(path + '/resources/' + name + '.yaml')) as f:
    config = yaml.safe_load(f)

# Power curve arrays and metadata from the config
POW, V, P, HUB_HEIGHT = (
    config["POW"],        # turbine electrical power at each wind speed (e.g., in W)
    config["V"],          # wind speed bins corresponding to the power curve (m/s)
    config["P"],          # rated (nameplate) power of the turbine (same unit as POW)
    config["HUB_HEIGHT"]  # turbine hub height (m)
)

# Convert lists to NumPy arrays for fast vectorized operations
POW = np.array(POW)
V   = np.array(V)


def average_lat_long(ds, to_height=HUB_HEIGHT, from_height=10, wnd_shear_exp=0.143):
    """
    Compute area-mean wind speed over a lat/lon box and scale it from a reference
    height to the turbine hub height using the power law.

    Parameters
    ----------
    ds : xr.Dataset
        Dataset containing at least u10 and v10 (10 m wind components).
    to_height : float
        Target height to scale wind speed to (m), typically the hub height.
    from_height : float
        Source height of provided winds (m). ERA/ECMWF 10 m winds use 10 m.
    wnd_shear_exp : float
        Wind shear exponent for the power-law profile (default 1/7 ≈ 0.143).

    Returns
    -------
    xr.DataArray
        Area-averaged wind speed (at hub height) over the specified region,
        retaining the original non-spatial dimensions (e.g., time, number).
    """

    # Spatial subset (box selection). Note: depending on coordinate ordering,
    # slicing may need ascending or descending bounds.
    subset = ds.sel(
        latitude=slice(55.0, 53.0),
        longitude=slice(6.0, 10.0)
    )

    # Compute 10 m wind speed from u and v components
    subset['windspeed'] = np.sqrt(subset["u10"]**2 + subset["v10"]**2)

    # Scale wind speed to hub height using an exponential power law:
    # U(z) = U(z0) * (z / z0) ** alpha
    subset["windspeed"] = subset['windspeed'] * (to_height / from_height) ** wnd_shear_exp

    # Return spatial mean across latitude and longitude
    return subset['windspeed'].mean(dim=("latitude", "longitude"))
    # If you prefer averaging over all of Germany instead, use:
    # return (ds["windspeed"].mean(dim=["latitude", "longitude"]))


def get_capacity_facors(data: xr):
    """
    Convert wind speed to capacity factor (0–1) by applying the turbine power curve.
    Note: function name has a small typo; it returns capacity factors.

    Parameters
    ----------
    data : xr.DataArray
        Wind speed at hub height.

    Returns
    -------
    xr.DataArray
        Capacity factor time series, same shape as input (0–1).
    """

    # Interpolation kernel: map wind speed -> normalized power (POW / P)
    def apply_power_curve(da):
        return np.interp(da, V, POW / P)  # normalized power output

    # Apply elementwise across the DataArray (works with Dask for parallelism)
    cf = xr.apply_ufunc(
        apply_power_curve,
        data,
        input_core_dims=[[]],
        output_core_dims=[[]],
        output_dtypes=[data.dtype],
        dask="parallelized",
    )

    # Name and metadata
    cf.name = "capacity_factor"
    return cf


def get_pypsa_export_structure(data: xr):
    """
    Convert capacity factor xarray data into a list of pandas DataFrames, one per
    turbine/asset index, with a fixed 3-hourly time index suitable for PyPSA-style
    CSV export.

    Parameters
    ----------
    data : xr.DataArray
        Capacity factor data with a 'number' dimension and chunks over 'valid_time'.

    Returns
    -------
    list[pd.DataFrame]
        One DataFrame per 'number' with a column 'cf_offwind_{i+1}' aligned
        to the target time index, linearly interpolated where necessary.
    """

    cf = []

    # Iterate over turbines/assets referenced by the 'number' coordinate
    for i in range(0, data.number[-1].values):
        # Target regular time index (adjust end as needed)
        index = pd.date_range(
            start="2019-01-01 00:00",
            end="2020-01-02 00:00",  # TODO: set according to data coverage
            freq="3h"
        )

        # Initialize empty DataFrame for this asset
        df = pd.DataFrame(index=index)
        df[f'cf_offwind_{i+1}'] = np.NaN

        # Fill from each chunk in 'data' (assumes iteration over data yields chunks)
        for data_chunk in data:
            dt = data_chunk['valid_time'].values
            # Convert times to string index matching the target frequency
            dt_str = pd.to_datetime(dt).strftime("%Y-%m-%d %H:%M:%S")
            # Assign the capacity factor values for turbine i at those times
            df.loc[dt_str, f'cf_offwind_{i+1}'] = data_chunk[i].values

        # Fill gaps by linear interpolation along the time index
        df[f'cf_offwind_{i+1}'] = df[f'cf_offwind_{i+1}'].interpolate(method="linear")

        cf.append(df)

    return cf


def prepare_dataset_wind():
    """
    End-to-end pipeline:
      1) Open all GRIB files from the specified folder (u10, v10).
      2) Compute area-mean hub-height wind speed time series.
      3) Map wind speed to capacity factor via the turbine power curve.
      4) Reshape into PyPSA export-friendly DataFrames.

    Returns
    -------
    list[pd.DataFrame]
        List of per-asset capacity factor time series DataFrames.
    """

    liste = []

    # Loop through ECMWF GRIB files (u10 = 165, v10 = 166)
    for file_name in glob.glob("/Users/mick/Documents/GitHub/masterthesis-mick/Wetterdaten/ECWMF_Data/*.grb"):
        ds = xr.open_dataset(
            file_name,
            engine="cfgrib",
            backend_kwargs={
                "filter_by_keys": {
                    "paramId": [165, 166],  # u10 and v10
                }
            }
        )

        # Area-mean hub-height wind speed for the region of interest
        ds = average_lat_long(ds)

        # Add a time dimension using the file's dataset time (as string)
        ds.expand_dims(time=[str(ds['time'].values)[:10]])

        liste.append(ds)

    # Concatenate along time and ensure chronological order
    data = xr.concat(liste, dim='time')
    data = data.sortby("time")

    # Convert wind speed -> capacity factor (0–1)
    data = get_capacity_facors(data)

    # Build PyPSA export structure (list of DataFrames)
    data = get_pypsa_export_structure(data)

    return data


In [19]:
data = prepare_dataset_wind()

for i in range(0,50):
    data[i] = data[i].loc[:'2019-12-31 22:00']

In [11]:
## read
# data = pd.read_pickle('/Users/mick/Documents/GitHub/masterthesis-mick/Wetterdaten/cappacity_factors_prepared/cf_offwind.pkl')

## write
# pd.to_pickle(data, "/Users/mick/Documents/GitHub/masterthesis-mick/Wetterdaten/cappacity_factors_prepared/cf_offwind.pkl")